In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, time
import pytz
import matplotlib.pyplot as plt

IST = pytz.timezone('Asia/Kolkata')
os.makedirs("strategy_results/trade_details", exist_ok=True)
os.makedirs("strategy_results/performance_metrics", exist_ok=True)
os.makedirs("strategy_results/charts", exist_ok=True)

TIMEFRAMES = {
    '3min': '3min',
    '5min': '5min',
    '15min': '15min',
    '30min': '30min',
    '1H': '1h',
    '1D': '1D'
}

# Market time
MARKET_START = time(9, 15)
MARKET_END = time(15, 30)

# Nifty50 symbols
nifty50_symbols = ["ADANIPORTS", "ASIANPAINT", "AXISBANK", "BAJAJ-AUTO", "BAJFINANCE", "BAJAJFINSV", 
                "BPCL", "BHARTIARTL", "BRITANNIA", "CIPLA", "COALINDIA", "DIVISLAB", "DRREDDY", "EICHERMOT",
                "GRASIM", "HCLTECH", "HDFCBANK", "HDFCLIFE", "HEROMOTOCO", "HINDALCO", "HINDUNILVR", 
                "ICICIBANK", "ITC", "INDUSINDBK", "INFY", "JSWSTEEL", "KOTAKBANK", "LT", "M&M", "MARUTI",
                "NTPC", "NESTLEIND", "ONGC", "POWERGRID", "RELIANCE", "SBILIFE", "SHREECEM", "SBIN",
                "SUNPHARMA", "TCS", "TATACONSUM", "TATAMOTORS", "TATASTEEL", "TECHM", "TITAN", "ULTRACEMCO", 
                "UPL", "WIPRO", "ZEEL", "LTIM"]


## Indicators

In [2]:

def calculate_ema(series, period):
    return series.ewm(span=period, adjust=False).mean()

def calculate_smi(df, k_period=40, d_period=20, ema_period=10):
    highest_high = df['High'].rolling(window=k_period, min_periods=k_period).max()
    lowest_low = df['Low'].rolling(window=k_period, min_periods=k_period).min()
    midpoint = (highest_high + lowest_low) / 2
    range_ = highest_high - lowest_low
    range_ = range_.replace(0, np.nan)
    min_range = df['Close'].mean() * 0.001
    range_ = range_.fillna(min_range)
    smi_k_raw = 100 * (df['Close'] - midpoint) / (0.5 * range_)
    smi_k = smi_k_raw.ewm(span=ema_period, adjust=False).mean()
    smi_d = smi_k.ewm(span=d_period, adjust=False).mean()
    return smi_k, smi_d

def calculate_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = ranges.max(axis=1)
    atr = true_range.rolling(period).mean()
    return atr

def calculate_supertrend(df, atr_period=10, multiplier=3):
    atr = calculate_atr(df, atr_period)
    hl2 = (df['High'] + df['Low']) / 2
    upper_band = hl2 + (multiplier * atr)
    lower_band = hl2 - (multiplier * atr)
    supertrend = pd.Series(index=df.index, dtype=float)
    direction = pd.Series(index=df.index, dtype=int)
    for i in range(1, len(df)):
        if df['Close'].iloc[i] > upper_band.iloc[i-1]:
            direction.iloc[i] = 1
        elif df['Close'].iloc[i] < lower_band.iloc[i-1]:
            direction.iloc[i] = -1
        else:
            direction.iloc[i] = direction.iloc[i-1]
            if direction.iloc[i] == 1:
                lower_band.iloc[i] = max(lower_band.iloc[i], lower_band.iloc[i-1])
            else:
                upper_band.iloc[i] = min(upper_band.iloc[i], upper_band.iloc[i-1])
        supertrend.iloc[i] = lower_band.iloc[i] if direction.iloc[i] == 1 else upper_band.iloc[i]
    return supertrend


## Filter and resample data

In [3]:
def filter_market_hours(df):
    """Filter market hours (9:15 AM to 3:30 PM IST)"""
    df = df.copy()
    df['time'] = df.index.time
    mask = (df['time'] >= MARKET_START) & (df['time'] <= MARKET_END)
    df = df[mask]
    df = df.drop('time', axis=1)
    return df

def resample_data(df, timeframe):
    """Resample data to specified timeframe considering market hours"""
    if timeframe == '1min':
        return df.copy()
    
    resampled = df.resample(timeframe, closed='right', label='right').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum'
    }).dropna()
    
    return resampled


## Strategy

In [4]:
def run_strategy(df, commission=0.0005, slippage=0.0005):
    if df is None or len(df) < 30:
        return None
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    df['EMA_3'] = calculate_ema(df['Close'], 3)
    df['EMA_30'] = calculate_ema(df['Close'], 30)
    df['SMI_K'], df['SMI_D'] = calculate_smi(df)
    df['Supertrend'] = calculate_supertrend(df)

    df['Signal'] = 0
    df['Position'] = 0
    df['Entry_Price'] = np.nan
    df['Exit_Price'] = np.nan
    df['P/L'] = 0.0
    df['Trade_Active'] = False
    df['Trade_Duration'] = 0
    df['Commission'] = 0.0
    df['Slippage'] = 0.0
    df['Net_P/L'] = 0.0
    df['Exit_Reason'] = ''
    df['Trade_ID'] = 0
    df['EMA_Cross'] = 0
    df['Crossover_Detected'] = False
    df['Entry_Time'] = None
    df['Exit_Time'] = None
    trade_counter = 1

    # EMA crossovers
    for i in range(1, len(df)):
        ema_distance = abs(df['EMA_3'].iloc[i] - df['EMA_30'].iloc[i])
        min_distance = df['Close'].iloc[i] * 0.001
        prev_above = df['EMA_3'].iloc[i-1] > df['EMA_30'].iloc[i-1]
        curr_above = df['EMA_3'].iloc[i] > df['EMA_30'].iloc[i]

        if prev_above != curr_above and ema_distance >= min_distance:
            df.at[df.index[i], 'Crossover_Detected'] = True
            if curr_above:
                df.at[df.index[i], 'EMA_Cross'] = 1  # Long
            else:
                df.at[df.index[i], 'EMA_Cross'] = -1  # Short

    for i in range(1, len(df)):
        current = df.index[i]
        prev = df.index[i-1]

        if df.loc[prev, 'Crossover_Detected'] and not df.loc[prev, 'Trade_Active']:
            signal = df.loc[prev, 'EMA_Cross']
            
            if signal == 1:
                print(f"Long signal detected at {current} - EMA_3: {df.loc[prev, 'EMA_3']:.2f}, EMA_30: {df.loc[prev, 'EMA_30']:.2f}")
            elif signal == -1:
                print(f"Short signal detected at {current} - EMA_3: {df.loc[prev, 'EMA_3']:.2f}, EMA_30: {df.loc[prev, 'EMA_30']:.2f}")
                
            entry_price = float(df.loc[current, 'Open'])
            position = int(signal)
            slippage_amt = entry_price * slippage
            commission_amt = entry_price * commission            
            df.at[current, 'Signal'] = signal
            df.at[current, 'Position'] = position
            df.at[current, 'Entry_Price'] = entry_price
            df.at[current, 'Trade_Active'] = True
            df.at[current, 'Slippage'] = slippage_amt
            df.at[current, 'Commission'] = commission_amt
            df.at[current, 'Net_P/L'] = -slippage_amt - commission_amt
            df.at[current, 'Trade_ID'] = trade_counter
            df.at[current, 'Entry_Time'] = current
            trade_counter += 1

        if df.loc[prev, 'Trade_Active']:
            pos = df.loc[prev, 'Position']
            entry = df.loc[prev, 'Entry_Price']
            df.at[current, 'Trade_Duration'] = df.loc[prev, 'Trade_Duration'] + 1
            df.at[current, 'Trade_ID'] = df.loc[prev, 'Trade_ID']
            df.at[current, 'Entry_Time'] = df.loc[prev, 'Entry_Time']
            prev_smi = df.loc[prev, 'SMI_K']
            curr_smi = df.loc[current, 'SMI_K']
            exit_price = df.loc[current, 'Open']
            exit_reason = None

            # exit conditions
            if pos == 1 and prev_smi > 0 and curr_smi < 0:
                exit_reason = 'SMI_Exit'
            elif pos == -1 and prev_smi < 0 and curr_smi > 0:
                exit_reason = 'SMI_Exit'
            elif pos == 1 and df.loc[current, 'Signal'] == -1:
                exit_reason = 'Reverse_Short'
            elif pos == -1 and df.loc[current, 'Signal'] == 1:
                exit_reason = 'Reverse_Long'
            elif pos == 1 and df.loc[current, 'Close'] < df.loc[current, 'Supertrend']:
                exit_reason = 'ST_Stop'
            elif pos == -1 and df.loc[current, 'Close'] > df.loc[current, 'Supertrend']:
                exit_reason = 'ST_Stop'

            if exit_reason:
                pl = exit_price - entry if pos == 1 else entry - exit_price
                slippage_amt = exit_price * slippage
                commission_amt = exit_price * commission
                net_pl = pl - slippage_amt - commission_amt
                df.at[current, 'Position'] = 0
                df.at[current, 'Exit_Price'] = exit_price
                df.at[current, 'P/L'] = pl
                df.at[current, 'Slippage'] = slippage_amt
                df.at[current, 'Commission'] = commission_amt
                df.at[current, 'Net_P/L'] = net_pl
                df.at[current, 'Trade_Active'] = False
                df.at[current, 'Exit_Reason'] = exit_reason
                df.at[current, 'Exit_Time'] = current

                if exit_reason.startswith('Reverse'):
                    new_pos = -1 if pos == 1 else 1
                    entry_price = exit_price
                    slippage_amt = entry_price * slippage
                    commission_amt = entry_price * commission
                    trade_counter += 1                    
                    df.at[current, 'Position'] = new_pos
                    df.at[current, 'Entry_Price'] = entry_price
                    df.at[current, 'Trade_Active'] = True
                    df.at[current, 'Slippage'] = slippage_amt
                    df.at[current, 'Commission'] = commission_amt
                    df.at[current, 'Net_P/L'] -= slippage_amt + commission_amt
                    df.at[current, 'Trade_ID'] = trade_counter
                    df.at[current, 'Entry_Time'] = current

    return df


## Performance metrics

In [5]:

def calculate_performance_metrics(df, initial_capital=100000):
    # Performance metrics
    if df is None or len(df) == 0:
        return None    
    trades = df[df['P/L'] != 0].copy()
    winning_trades = trades[trades['P/L'] > 0]
    losing_trades = trades[trades['P/L'] < 0]
    
    # Equity curve
    df['Cumulative_P/L'] = df['Net_P/L'].cumsum()
    df['Equity'] = initial_capital + df['Cumulative_P/L']
    
    # Drawdown
    df['Peak'] = df['Equity'].cummax()
    df['Drawdown'] = (df['Equity'] - df['Peak']) / df['Peak'] * 100
    # Returns
    df['Returns'] = df['Equity'].pct_change()
    
    # Performance metrics
    metrics = {
        'Total_Trades': len(trades),
        'Winning_Trades': len(winning_trades),
        'Losing_Trades': len(losing_trades),
        'Win_Rate': len(winning_trades) / len(trades) if len(trades) > 0 else 0,
        'Profit_Factor': abs(winning_trades['P/L'].sum() / losing_trades['P/L'].sum()) if len(losing_trades) > 0 else float('inf'),
        'Average_Win': winning_trades['P/L'].mean() if len(winning_trades) > 0 else 0,
        'Average_Loss': losing_trades['P/L'].mean() if len(losing_trades) > 0 else 0,
        'Largest_Win': winning_trades['P/L'].max() if len(winning_trades) > 0 else 0,
        'Largest_Loss': losing_trades['P/L'].min() if len(losing_trades) > 0 else 0,
        'Average_Trade_Duration': trades['Trade_Duration'].mean() if len(trades) > 0 else 0,
        'Total_Net_Profit': df['Net_P/L'].sum(),
        'Max_Drawdown': df['Drawdown'].min(),
        'Sharpe_Ratio': np.sqrt(252) * df['Returns'].mean() / df['Returns'].std() if len(df) > 1 else 0,
        'Sortino_Ratio': np.sqrt(252) * df['Returns'].mean() / df['Returns'][df['Returns'] < 0].std() if len(df[df['Returns'] < 0]) > 0 else 0,
        'Expectancy': (len(winning_trades) * winning_trades['P/L'].mean() + len(losing_trades) * losing_trades['P/L'].mean()) / len(trades) if len(trades) > 0 else 0
    }
    
    return metrics, df

def format_duration(duration, timeframe):
    if timeframe == '1D':
        days = int(duration)
        return f"{days} days" if days != 1 else "1 day"
    elif timeframe == '1H':
        hours = int(duration)
        return f"{hours} hours" if hours != 1 else "1 hour"
    else:
        minutes = int(duration)
        if timeframe == '3min':
            minutes *= 3
        elif timeframe == '5min':
            minutes *= 5
        elif timeframe == '15min':
            minutes *= 15
        elif timeframe == '30min':
            minutes *= 30
        return f"{minutes} min"


## Plot charts

In [6]:
def plot_performance_charts(df, ticker, timeframe, metrics):
    plt.style.use('default')
    fig = plt.figure(figsize=(15, 12))
    ax1 = plt.subplot(3, 1, 1)
    ax1.plot(df.index, df['Close'], label='Price', color='black', linewidth=1)
    ax1.plot(df.index, df['EMA_3'], label='EMA 3', color='blue', alpha=0.5)
    ax1.plot(df.index, df['EMA_30'], label='EMA 30', color='red', alpha=0.5)
    long_entries = df[(df['Signal'] == 1) & (df['Position'] == 1)]
    long_exits = df[(df['Position'] == 1) & (df['Exit_Reason'] != '')]
    ax1.scatter(long_entries.index, long_entries['Open'], marker='^', color='g', s=100, label='Long Entry')
    ax1.scatter(long_exits.index, long_exits['Open'], marker='v', color='darkgreen', s=100, label='Long Exit')
    short_entries = df[(df['Signal'] == -1) & (df['Position'] == -1)]
    short_exits = df[(df['Position'] == -1) & (df['Exit_Reason'] != '')]
    ax1.scatter(short_entries.index, short_entries['Open'], marker='v', color='r', s=100, label='Short Entry')
    ax1.scatter(short_exits.index, short_exits['Open'], marker='^', color='darkred', s=100, label='Short Exit')
    ax1.plot(df.index, df['Supertrend'], label='Supertrend', color='purple', alpha=0.7)
    ax1.set_title(f'{ticker} - {timeframe} Price with Trades')
    ax1.legend()
    ax1.grid(True)
    
    # Equity curve
    ax2 = plt.subplot(3, 1, 2)
    ax2.plot(df.index, df['Equity'], label='Equity Curve', color='blue')
    ax2.set_title('Equity Curve')
    ax2.grid(True)
    
    # Drawdown
    ax3 = plt.subplot(3, 1, 3)
    ax3.fill_between(df.index, df['Drawdown'], 0, color='red', alpha=0.3)
    ax3.set_title('Drawdown')
    ax3.grid(True)
    
    plt.tight_layout()
    plt.savefig(f'strategy_results/charts/{ticker}_{timeframe}_performance.png')
    plt.close()


## Save trades

In [7]:
def save_trade_details(df, ticker, timeframe):
    if df is None or 'Trade_ID' not in df.columns:
        return

    entries = df[df['Signal'] != 0].copy()
    exits = df[df['Exit_Reason'] != ''].copy()
    
    if len(entries) == 0 or len(exits) == 0:
        return
    
    trade_details = []
    for trade_id in df['Trade_ID'].unique():
        if trade_id == 0:
            continue
            
        trade_data = df[df['Trade_ID'] == trade_id]
        if len(trade_data) == 0:
            continue
            
        entry = trade_data.iloc[0]
        exit = trade_data.iloc[-1] if len(trade_data) > 1 else entry
        
        if pd.isna(entry['Entry_Price']) or pd.isna(exit['Exit_Price']):
            continue
            
        entry_date = entry['Entry_Time'].strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(entry['Entry_Time']) else trade_data.index[0].strftime('%Y-%m-%d %H:%M:%S%z')
        exit_date = exit['Exit_Time'].strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(exit['Exit_Time']) else trade_data.index[-1].strftime('%Y-%m-%d %H:%M:%S%z')
        
        trade_info = {
            'Trade_ID': trade_id,
            'Entry_Date': entry_date,
            'Exit_Date': exit_date,
            'Position': 'Long' if entry['Position'] == 1 else 'Short',
            'Entry_Price': entry['Entry_Price'],
            'Exit_Price': exit['Exit_Price'],
            'P/L': exit['P/L'],
            'Net_P/L': exit['Net_P/L'],
            'Duration': format_duration(exit['Trade_Duration'], timeframe),
            'Exit_Reason': exit['Exit_Reason'],
            'Entry_Candle_Open': entry['Open'],
            'Entry_Candle_Close': entry['Close'],
            'Exit_Candle_Open': exit['Open'],
            'Exit_Candle_Close': exit['Close'],
            'EMA_3_Entry': entry['EMA_3'],
            'EMA_30_Entry': entry['EMA_30'],
            'SMI_K_Entry': entry['SMI_K'],
            'SMI_D_Entry': entry['SMI_D'],
            'Supertrend_Entry': entry['Supertrend']
        }
        trade_details.append(trade_info)
    
    trades_df = pd.DataFrame(trade_details)
    trades_df.to_csv(f'strategy_results/trade_details/{ticker}_{timeframe}_trades.csv', index=False)


## Run

In [8]:
def run_backtest_for_timeframe(timeframe):
    results = []
    
    for ticker in nifty50_symbols:
        print(f"\nProcessing {ticker} {timeframe}...")
        
        try:
            df = pd.read_csv(f"nifty50_data/{ticker}_1min_1years_data.csv")
            df['date'] = pd.to_datetime(df['date'])
            df = df.set_index('date')            
            df = df.rename(columns={
                'open': 'Open',
                'high': 'High',
                'low': 'Low',
                'close': 'Close',
                'volume': 'Volume'
            })            
            df = filter_market_hours(df)
            df = resample_data(df, TIMEFRAMES[timeframe])
            
            if df is None or len(df) == 0:
                print(f"  Skipping {ticker} - No data available")
                continue            
            result_df = run_strategy(df)
            if result_df is None:
                print(f"  Skipping {ticker} - Strategy execution failed")
                continue            
            metrics, equity_df = calculate_performance_metrics(result_df)
            
            # Save results
            # result_df.to_csv(f"strategy_results/{ticker}_{timeframe}_results.csv")
            save_trade_details(result_df, ticker, timeframe)
            plot_performance_charts(equity_df, ticker, timeframe, metrics)
            metrics['Ticker'] = ticker
            metrics['Timeframe'] = timeframe
            results.append(metrics)
            
            print(f"  Trades: {metrics['Total_Trades']}")
            print(f"  Win Rate: {metrics['Win_Rate']:.1%}")
            print(f"  Net PnL: {metrics['Total_Net_Profit']:.2f}")
            
        except Exception as e:
            print(f"  Error processing {ticker}: {str(e)}")
            continue
    
    return results

def generate_summary_report(all_results):
    if not all_results:
        print("No results to summarize")
        return
    
    combined_results = []
    for timeframe_results in all_results.values():
        combined_results.extend(timeframe_results)
    
    summary_df = pd.DataFrame(combined_results)
    # Save summary
    summary_df.to_csv("strategy_results/performance_metrics/summary_results.csv", index=False)
    timeframe_comparison = summary_df.groupby('Timeframe').agg({
        'Total_Trades': 'mean',
        'Win_Rate': 'mean',
        'Profit_Factor': 'mean',
        'Total_Net_Profit': 'mean',
        'Sharpe_Ratio': 'mean',
        'Sortino_Ratio': 'mean',
        'Max_Drawdown': 'mean'
    }).round(4)
    
    timeframe_comparison.to_csv("strategy_results/performance_metrics/timeframe_comparison.csv")
    
    print("\n========= TIMEFRAME COMPARISON =========")
    print(timeframe_comparison)

def run_backtest():
    # Run backtest for all timeframes
    all_results = {}
    for timeframe in TIMEFRAMES.keys():
        print(f"\n========= RUNNING BACKTEST FOR TIMEFRAME: {timeframe} =========")
        results = run_backtest_for_timeframe(timeframe)
        all_results[timeframe] = results    
    generate_summary_report(all_results)

if __name__ == "__main__":
    run_backtest()


========= RUNNING BACKTEST FOR TIMEFRAME: 3min =========

Processing ADANIPORTS 3min...
Long signal detected at 2024-04-30 09:39:00+05:30 - EMA_3: 1323.38, EMA_30: 1321.17
Short signal detected at 2024-05-02 14:57:00+05:30 - EMA_3: 1336.01, EMA_30: 1338.22
Short signal detected at 2024-05-03 09:27:00+05:30 - EMA_3: 1336.85, EMA_30: 1339.15
Long signal detected at 2024-05-06 11:54:00+05:30 - EMA_3: 1284.97, EMA_30: 1283.51
Short signal detected at 2024-05-06 13:18:00+05:30 - EMA_3: 1284.97, EMA_30: 1287.53
Short signal detected at 2024-05-07 10:06:00+05:30 - EMA_3: 1292.84, EMA_30: 1294.22
Short signal detected at 2024-05-08 09:18:00+05:30 - EMA_3: 1283.25, EMA_30: 1286.07
Short signal detected at 2024-05-08 10:36:00+05:30 - EMA_3: 1284.97, EMA_30: 1286.77
Long signal detected at 2024-05-09 09:18:00+05:30 - EMA_3: 1280.09, EMA_30: 1278.49
Long signal detected at 2024-05-10 09:21:00+05:30 - EMA_3: 1253.56, EMA_30: 1250.13
Long signal detected at 2024-05-10 13:36:00+05:30 - EMA_3: 1272.3